# QC-Py-41 : Paper Trading IBKR - SP500 Momentum

> **Objectif** : Workflow backtest -> paper trading sur equities via Interactive Brokers.

## Prerequis
- QC-Py-09 (Order Types) : types d'ordres et leur execution
- QC-Py-12 (Backtesting Analysis) : lecture de resultats de backtest
- QC-Py-27 (Production Deployment) : workflow de deploiement live
- QC-Py-40 (Paper Trading Binance) : concepts du paper trading

## Plan du notebook
1. Interactive Brokers comme broker multi-asset
2. Strategie momentum cross-section SP500
3. Backtest historique
4. Configuration IBKR et prerequis TWS
5. Deploy en paper trading
6. Monitoring
7. Exercice : ajouter un filtre de volatilite

In [ ]:
import os
import json
import time
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Configuration matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print(f"QC-Py-41 Paper Trading IBKR - {datetime.now().strftime('%Y-%m-%d')}")

## 1. Interactive Brokers : Broker Multi-Asset

Interactive Brokers (IBKR) est le broker le plus utilise en trading algorithmique institutionnel :

| Caracteristique | Detail |
|----------------|--------|
| Asset classes | Equities, Options, Forex, Futures, Future Options |
| Order types | Market, Limit, Stop, StopLimit, MarketOnOpen/Close, + custom |
| Order updates | Supportes (modifier sans annuler) |
| Donnees | Tick, Second, Minute, Hour, Daily |
| Paper trading | TWS Paper (port 7497) |

### Prerequis specifiques IBKR

1. **Compte IBKR** : ouverture en ligne (KYC, 1-3 jours)
2. **TWS ou IB Gateway** : doit tourner en continu
3. **Configuration API** : ActiveX/Socket active, port 7497 (paper) / 7496 (live)
4. **Redemarrage hebdomadaire** : IBKR exige un restart avec 2FA chaque semaine

In [ ]:
# Comparaison Binance vs IBKR pour le paper trading

comparison = pd.DataFrame({
    'Aspect': ['Assets', 'Credentials', 'Prerequis', 'Order Updates',
               'Fees simulees', 'Resolution min', 'Paper env'],
    'Binance': ['Crypto spot', 'Testnet (gratuit)', 'Aucun', 'Non',
                '0.1%', 'Minute', 'Spot Test Network'],
    'IBKR': ['Multi-asset', 'Compte paper IBKR', 'TWS/Gateway', 'Oui',
             'Variable ($0.005/share)', 'Tick', 'TWS Paper (7497)']
})

print("Comparaison Binance vs IBKR:")
print(comparison.to_string(index=False))

## 2. Strategie : Momentum Cross-Section SP500

La strategie momentum selectionne les actions du SP500 avec la meilleure performance relative sur les 12 derniers mois :

- **Univers** : Top-50 SP500 par capitalisation
- **Signal** : Return sur 252 jours (12 mois)
- **Selection** : Top 10 momentum (les plus performants)
- **Rebalancement** : Mensuel
- **Poids** : Equiponderable (10% par position)

In [ ]:
# Algorithme LEAN : Momentum SP500 via IBKR

LEAN_ALGORITHM = '''# region imports
from AlgorithmImports import *
# endregion


class SP500MomentumIBKR(QCAlgorithm):
    """
    Cross-sectional momentum on SP500 top-50.
    IBKR paper trading demonstration.

    Logic:
    - Rank top-50 SP500 stocks by 12-month return
    - Hold top 10 momentum winners
    - Rebalance monthly, equal weight
    """

    def initialize(self):
        self.set_start_date(2018, 1, 1)
        self.set_end_date(2024, 12, 31)
        self.set_cash(100000)

        # Brokerage IBKR
        self.set_brokerage_model(
            BrokerageName.INTERACTIVE_BROKERS_BROKERAGE,
            AccountType.MARGIN
        )

        # SP500 top-50 par market cap (snapshot statique)
        self.tickers = [
            "AAPL", "MSFT", "NVDA", "AMZN", "GOOGL",
            "META", "BRK.B", "LLY", "TSM", "AVGO",
            "JPM", "XOM", "UNH", "V", "PG",
            "JNJ", "MA", "HD", "COST", "ABBV",
            "MRK", "CRM", "ORCL", "WMT", "BAC",
            "NFLX", "AMD", "ADBE", "CVX", "KO",
            "PEP", "TMO", "CSCO", "MCD", "ABT",
            "INTC", "ACN", "DHR", "VZ", "WFC",
            "TXN", "NEE", "PM", "NKE", "LIN",
            "RTX", "BMY", "QCOM", "HON", "UPS",
        ]

        self.symbols = []
        for ticker in self.tickers:
            equity = self.add_equity(ticker, Resolution.DAILY)
            self.symbols.append(equity.symbol)

        # Benchmark
        self.set_benchmark("SPY")
        self.set_warm_up(252, Resolution.DAILY)

        # Parametres
        self.num_holdings = 10
        self.momentum_period = 252  # 12 mois
        self.rebalance_freq = 30    # jours
        self.last_rebalance = -999

    def on_data(self, data):
        if self.is_warming_up:
            return

        # Rebalancement mensuel
        days_since = self.time.day if hasattr(self, '_last_day') else 0
        if self.time.day < 5:  # Debut du mois
            if self.last_rebalance < self.time.month:
                self._rebalance(data)
                self.last_rebalance = self.time.month

    def _rebalance(self, data):
        # Calculer les retours momentum
        momentum_scores = {}
        for sym in self.symbols:
            if sym not in data or data[sym] is None:
                continue
            history = self.history(sym, self.momentum_period, Resolution.DAILY)
            if history is None or len(history) < self.momentum_period * 0.8:
                continue
            closes = history["close"] if isinstance(history, pd.DataFrame) else history.close
            ret = (closes.iloc[-1] / closes.iloc[0]) - 1
            momentum_scores[sym] = ret

        # Selectionner top N momentum
        sorted_momentum = sorted(momentum_scores.items(), key=lambda x: x[1], reverse=True)
        top_n = sorted_momentum[:self.num_holdings]
        target_symbols = set(sym for sym, _ in top_n)

        # Liquidier les positions qui ne sont plus dans le top
        for sym in self.symbols:
            if self.portfolio[sym].invested and sym not in target_symbols:
                self.liquidate(sym)

        # Acheter les nouvelles positions
        weight = 1.0 / self.num_holdings
        for sym, ret in top_n:
            self.set_holdings(sym, weight)

        self.log(f"REBALANCE: Top-10 momentum, best={top_n[0][0].value} (+{top_n[0][1]:.1%})")

    def on_end_of_algorithm(self):
        final = self.portfolio.total_portfolio_value
        ret = (final - 100000) / 100000
        self.log(f"SP500 Momentum IBKR: Final=${final:,.2f}, Return={ret:.2%}")
'''

print("Algorithme LEAN SP500 Momentum pret.")
print(f"Taille: {len(LEAN_ALGORITHM)} caracteres")

## 3. Backtest Historique

La strategie momentum est bien documentee dans la litterature academique (Jegadeesh & Titman, 1993). Sur le SP500, le momentum 12 mois a historiquement genere un premium annualise de 5-8%.

In [ ]:
# Resultats de backtest simules (en pratique via read_backtest MCP QC)

backtest_results = {
    "strategy": "SP500 Momentum Top-10 (12m lookback)",
    "period": "2018-01-01 to 2024-12-31",
    "initial_capital": 100000,
    "metrics": {
        "Total Return": "+142%",
        "CAGR": "13.6%",
        "Sharpe Ratio": "0.95",
        "Max Drawdown": "-32.1%",
        "Win Rate (monthly)": "62%",
        "Total Trades": 480,
        "Avg Monthly Turnover": "40%",
        "Beta vs SPY": "1.05",
    },
    "benchmark": {
        "SPY Buy & Hold": "+118%",
        "Equal Weight Top-50": "+95%",
    }
}

print(f"=== Backtest: {backtest_results['strategy']} ===")
print(f"Periode: {backtest_results['period']}")
print(f"Capital initial: ${backtest_results['initial_capital']:,}")
print("\nMetriques:")
for k, v in backtest_results["metrics"].items():
    print(f"  {k}: {v}")
print("\nBenchmark:")
for k, v in backtest_results["benchmark"].items():
    print(f"  {k}: {v}")

In [ ]:
# Visualisation equity curve vs benchmark
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Equity curves simulees
np.random.seed(42)
dates = pd.date_range('2018-01-01', '2024-12-31', freq='W')

# Strategie momentum
mom_returns = np.random.normal(0.003, 0.035, len(dates))
mom_equity = 100000 * np.cumprod(1 + mom_returns)

# SPY benchmark
spy_returns = np.random.normal(0.002, 0.03, len(dates))
spy_equity = 100000 * np.cumprod(1 + spy_returns)

axes[0].plot(dates, mom_equity, color='#2196F3', linewidth=1.5, label='Momentum Top-10')
axes[0].plot(dates, spy_equity, color='#FF9800', linewidth=1.5, label='SPY', alpha=0.8)
axes[0].set_title('Equity Curve vs Benchmark')
axes[0].set_ylabel('Valeur Portfolio ($)')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=45)

# Drawdown
peak = np.maximum.accumulate(mom_equity)
drawdown = (mom_equity - peak) / peak * 100
axes[1].fill_between(dates, drawdown, 0, alpha=0.4, color='#F44336')
axes[1].set_title('Drawdown Momentum (%)')
axes[1].set_ylabel('Drawdown (%)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()
plt.close()

## 4. Configuration IBKR et Prerequis TWS

Contrairement a Binance, IBKR necessite une infrastructure locale :

### Checklist de configuration

1. Compte IBKR paper trading actif
2. TWS (Trader Workstation) telecharge et installe
3. Configuration API dans TWS :
   - Edit > Global Configuration > API > Settings
   - Cocher "Enable ActiveX and Socket Clients"
   - Port : **7497** (paper) / 7496 (live)
   - Deconnecter "Read-Only API"
4. IB Gateway (alternative a TWS, plus leger)
5. Redemarrage hebdomadaire avec 2FA

In [ ]:
# Configuration IBKR pour le deploiement via MCP QC

ibkr_config = {
    "brokerage": "InteractiveBrokersBrokerage",
    "paper_port": 7497,
    "live_port": 7496,
    "required_fields": [
        "ib-user-name",
        "ib-account",
        "ib-password",
        "ib-weekly-restart-utc-time",
    ],
    "weekly_restart": "04:00:00",  # Dimanche 04h UTC
}

# Deploy via MCP QC (reference)
deploy_reference = """
create_live_algorithm(
    model={
        "projectId": PROJECT_ID,
        "compileId": compile_id,
        "nodeId": "LN-xxxxx",  # L-MICRO node
        "versionId": "-1",
        "brokerage": {
            "id": "InteractiveBrokersBrokerage",
            "ib-user-name": "<username>",
            "ib-account": "DU12345",  # Paper account ID
            "ib-password": "<password>",
            "ib-weekly-restart-utc-time": "04:00:00"
        },
        "dataProviders": {
            "QuantConnectBrokerage": {
                "id": "QuantConnectBrokerage"
            }
        }
    }
)
"""

print("Configuration IBKR Paper Trading:")
for k, v in ibkr_config.items():
    print(f"  {k}: {v}")

## 5. Deploy en Paper Trading

Le deploiement IBKR suit le meme workflow que Binance, avec des credentials differents.

La difference principale : **IBKR necessite TWS qui tourne en local**. Le node live QC se connecte au TWS via l'API socket.

```
Internet → QC Node Live → API Socket → TWS (local) → IBKR Paper Market
```

In [ ]:
# Workflow complet de deploiement IBKR

deployment_steps = [
    {"step": 1, "action": "Creer le projet QC", "tool": "create_project()"},
    {"step": 2, "action": "Ecrire main.py", "tool": "update_file_contents()"},
    {"step": 3, "action": "Compiler", "tool": "create_compile() + read_compile()"},
    {"step": 4, "action": "Backtest validation", "tool": "create_backtest() + read_backtest()"},
    {"step": 5, "action": "Verifier TWS actif", "tool": "(manuel)"},
    {"step": 6, "action": "Deploy paper trading", "tool": "create_live_algorithm()"},
    {"step": 7, "action": "Monitorer", "tool": "read_live_algorithm()"},
]

print("Workflow de deploiement IBKR:")
for step in deployment_steps:
    print(f"  {step['step']}. {step['action']:<30s} [{step['tool']}]")

## 6. Monitoring

Les memes outils MCP QC que pour Binance s'appliquent : `read_live_algorithm`, `read_live_orders`, `read_live_logs`, `read_live_portfolio`.

Points specifiques IBKR a surveiller :
- **Connection TWS** : si TWS s'arrete, le node live perd la connexion
- **Weekly restart** : verifier que le redemarrage automatique fonctionne
- **Order fills** : les fills paper TWS peuvent differer legerement du live

In [ ]:
# Comparaison des ecarts backtest vs paper trading IBKR

ecarts_ibkr = pd.DataFrame({
    'Source': ['Slippage', 'Timing', 'Liquidite', 'Market impact', 'Connection'],
    'Impact estime': ['Faible', 'Moyen', 'Faible', 'Faible', 'Eleve (TWS)'],
    'Description': [
        'Fills paper TWS proches du live sur grosses caps',
        'Latence TWS + QC node, quelques ms',
        'SP500 top-50: tres liquide, pas de probleme',
        'Taille portfolio petite vs volume daily',
        'Risque de deconnexion si TWS plante'
    ]
})

print("Ecarts backtest vs paper trading (IBKR):")
print(ecarts_ibkr.to_string(index=False))

## 7. Exercice : Ajouter un Filtre de Volatilite

La strategie momentum pure peut souffrir pendant les regimes de marche volatils (ex: mars 2020). L'objectif est d'ajouter un filtre de volatilite pour reduire l'exposition pendant les periodes turbulentes.

### Approche proposee
- Calculer la volatilite realisee du SP500 sur 20 jours
- Si vol > seuil, reduire le nombre de positions (ou passer en cash)
- Parametres a optimiser : `vol_period`, `vol_threshold`, `reduction_factor`

In [ ]:
# TODO etudiant : implementer le filtre de volatilite

# Parametres du filtre
vol_config = {
    "vol_period": 20,       # Period pour le calcul de vol
    "vol_threshold": None,  # TODO etudiant : seuil de vol (ex: 0.30 = 30% annualisee)
    "reduction_factor": None,  # TODO etudiant : facteur de reduction (ex: 0.5 = moitie des positions)
}

# TODO etudiant : implementer la fonction ci-dessous
def apply_vol_filter(momentum_scores: Dict, spy_vol: float,
                     config: Dict) -> Dict:
    """
    Filtre les signaux momentum en fonction de la volatilite.
    
    Args:
        momentum_scores: {symbol: momentum_return}
        spy_vol: volatilite realisee du SP500
        config: parametres du filtre
    
    Returns:
        momentum_scores filtres
    """
    # TODO etudiant : implementer la logique
    # Indice 1 : si spy_vol > vol_threshold, garder uniquement
    #            les N * reduction_factor meilleures positions
    # Indice 2 : si spy_vol <= vol_threshold, ne rien changer
    pass  # TODO etudiant

# Indice : volatilite annualisee = std(returns daily) * sqrt(252)
# Exemple : si SPY daily returns std = 0.015, vol annualisee = 0.015 * sqrt(252) = 23.8%

print("Filtre de volatilite a implementer.")
print("Indice: volatilite annualisee = std(daily_returns) * sqrt(252)")
print(f"Config actuelle: {vol_config}")

In [ ]:
# Question de reflexion
questions = [
    "1. Pourquoi le momentum fonctionne-t-il historiquement sur le SP500 ?",
    "2. Quel est le risque principal d'une strategie momentum en paper trading ?",
    "3. Comment le filtre de volatilite peut-il ameliorer le ratio de Sharpe ?",
    "4. Quelle est la difference de comportement entre Binance et IBKR en paper trading ?",
]

# TODO etudiant : repondre aux questions dans une nouvelle cellule markdown
for q in questions:
    print(q)
print("\nTODO: Creez une cellule markdown ci-dessous pour vos reponses.")

## Conclusion

Ce notebook a couvert le workflow paper trading sur equities via IBKR :

1. **IBKR** : broker multi-asset, prerequis TWS, configuration API
2. **Strategie** : momentum cross-section SP500 (top-10, 12 mois)
3. **Backtest** : validation sur 7 ans, comparison avec SPY
4. **Deploy** : workflow MCP QC avec credentials IBKR
5. **Monitoring** : surveillance et ecarts backtest/paper
6. **Exercice** : filtre de volatilite pour ameliorer la robustesse

### References
- Jegadeesh & Titman (1993), "Returns to Buying Winners and Selling Losers"
- [QC IBKR Brokerage Docs](https://www.quantconnect.com/docs/v2/our-platform/live-trading/brokerages/interactive-brokers)
- [QC Paper Trading](https://www.quantconnect.com/docs/v2/our-platform/live-trading/brokerages/quantconnect-paper-trading)